<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 4


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Описание задачи: 
Создать базовый класс Product в C#, который будет представлять информацию о 
продуктах.  На  основе  этого  класса  разработать  2-3  производных  класса, 
демонстрирующих принципы наследования и полиморфизма. В каждом из классов 
должны быть реализованы новые атрибуты и методы, а также переопределены 
некоторые методы базового класса для демонстрации полиморфизма. 
Требования к базовому классу Product: 

• Атрибуты: Название (Name), Цена (Price), Производитель (Manufacturer). 

• Методы: 

    o GetInfo(): метод для получения информации о продукте в виде строки. 

    o Discount(): метод для применения скидки к цене продукта.

    o Display(): метод для отображения информации о продукте на экране.
     
Требования к производным классам: 
1. Электроника  (Electronics):  Должен  содержать  дополнительные  атрибуты, 
такие как Гарантийный срок (WarrantyPeriod). Метод Discount() должен быть 
переопределен  для  добавления  логики  учета  гарантийного  срока  при 
применении скидки. 
2. Одежда (Clothing): Должен содержать дополнительные атрибуты, такие как 
Размер (Size). Метод Display() должен быть переопределен для добавления 
информации о размере при отображении информации о продукте. 
3. Книги  (Books) (если  требуется  третий  класс):  Должен  содержать 
дополнительные атрибуты, такие как Автор (Author). Метод GetInfo() должен 
быть  переопределен  для  включения  информации  об  авторе  в  описании 
продукта

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [3]:
using System;
using System.Collections.Generic;
using System.Linq;

public delegate void ProductEventHandler(Product product, string message);
public delegate void DiscountAppliedHandler(Product product, decimal oldPrice, decimal newPrice);
public delegate bool ProductFilter(Product product);

public interface IDiscountable
{
    void ApplyDiscount(decimal percentage);
    bool IsDiscountAvailable();
}

public interface IShippable
{
    decimal CalculateShippingCost();
    string GetShippingInfo();
}

public interface IStockable
{
    void UpdateStock(int quantity);
    bool IsInStock();
    int GetStockLevel();
}

public abstract class Product : IDiscountable, IShippable, IStockable
{
    public event ProductEventHandler ProductUpdated;
    public event DiscountAppliedHandler DiscountApplied;
    public static event ProductEventHandler ProductCreated;

    public string Name { get; set; }
    public decimal Price { get; set; }
    public string Manufacturer { get; set; }
    public string Description { get; set; }
    public string SKU { get; set; }
    public DateTime CreatedDate { get; set; }
    public int StockQuantity { get; set; }
    public string Category { get; set; }
    public decimal Weight { get; set; }
    public bool IsActive { get; set; }
    public List<string> Tags { get; set; }
    public Dictionary<string, string> Specifications { get; set; }
    public int ViewCount { get; set; }
    public DateTime LastUpdated { get; set; }

    protected Product(string name, decimal price, string manufacturer)
    {
        Name = name;
        Price = price;
        Manufacturer = manufacturer;
        Description = string.Empty;
        SKU = GenerateSKU();
        CreatedDate = DateTime.Now;
        LastUpdated = DateTime.Now;
        StockQuantity = 0;
        Category = "General";
        Weight = 0;
        IsActive = true;
        ViewCount = 0;
        Tags = new List<string>();
        Specifications = new Dictionary<string, string>();
        ProductCreated?.Invoke(this, $"Создан новый продукт: {name}");
    }

    public virtual string GetInfo()
    {
        return $"Название: {Name}\nЦена: {Price:C}\nПроизводитель: {Manufacturer}\n" +
               $"Описание: {Description}\nSKU: {SKU}\nКатегория: {Category}\n" +
               $"Теги: {string.Join(", ", Tags)}\nПросмотры: {ViewCount}";
    }

    public virtual void Discount(decimal percentage)
    {
        if (percentage > 0 && percentage <= 100)
        {
            decimal oldPrice = Price;
            decimal discountAmount = Price * (percentage / 100);
            Price -= discountAmount;
            Console.WriteLine($"Применена скидка {percentage}% к {Name}. Новая цена: {Price:C}");
            DiscountApplied?.Invoke(this, oldPrice, Price);
            OnProductUpdated($"Применена скидка {percentage}%");
        }
    }

    public virtual void Display()
    {
        Console.WriteLine("=== ИНФОРМАЦИЯ О ПРОДУКТЕ ===");
        Console.WriteLine(GetInfo());
        Console.WriteLine($"В наличии: {((IStockable)this).IsInStock()}");
        Console.WriteLine($"Последнее обновление: {LastUpdated:dd.MM.yyyy HH:mm}");
        Console.WriteLine("=============================");
        IncrementViewCount();
    }

    public virtual void UpdateDescription(string newDescription)
    {
        Description = newDescription;
        OnProductUpdated("Обновлено описание");
    }

    public virtual decimal CalculateTax(decimal taxRate = 0.2m)
    {
        return Price * taxRate;
    }

    public virtual bool ValidateProduct()
    {
        return !string.IsNullOrEmpty(Name) && 
               !string.IsNullOrEmpty(Manufacturer) && 
               Price >= 0 && 
               StockQuantity >= 0;
    }

    public virtual string GetProductStatus()
    {
        return $"Статус: {(IsActive ? "Активен" : "Неактивен")}, " +
               $"Запас: {StockQuantity}, " +
               $"Вес: {Weight} кг, " +
               $"Просмотры: {ViewCount}";
    }

    public virtual void AddTag(string tag)
    {
        if (!Tags.Contains(tag))
        {
            Tags.Add(tag);
            OnProductUpdated($"Добавлен тег: {tag}");
        }
    }

    public virtual void RemoveTag(string tag)
    {
        if (Tags.Remove(tag))
        {
            OnProductUpdated($"Удален тег: {tag}");
        }
    }

    public virtual bool HasTag(string tag)
    {
        return Tags.Contains(tag);
    }

    public virtual void AddSpecification(string key, string value)
    {
        Specifications[key] = value;
        OnProductUpdated($"Добавлена спецификация: {key}");
    }

    public virtual string GetSpecification(string key)
    {
        return Specifications.ContainsKey(key) ? Specifications[key] : "Не указано";
    }

    public virtual void IncrementViewCount()
    {
        ViewCount++;
        if (ViewCount % 10 == 0)
        {
            OnProductUpdated($"Достигнуто {ViewCount} просмотров");
        }
    }

    public bool CheckAvailability()
    {
        return ((IStockable)this).IsInStock();
    }

    public void UpdateStock(int quantity)
    {
        ((IStockable)this).UpdateStock(quantity);
    }

    protected virtual void OnProductUpdated(string message)
    {
        LastUpdated = DateTime.Now;
        ProductUpdated?.Invoke(this, message);
    }

    void IDiscountable.ApplyDiscount(decimal percentage)
    {
        Console.WriteLine($"=== ЯВНАЯ РЕАЛИЗАЦИЯ IDiscountable ===");
        if (percentage > 0 && percentage <= 50)
        {
            decimal oldPrice = Price;
            Price -= Price * (percentage / 100);
            Console.WriteLine($"Скидка {percentage}% применена через интерфейс");
            Console.WriteLine($"Старая цена: {oldPrice:C}, Новая цена: {Price:C}");
            DiscountApplied?.Invoke(this, oldPrice, Price);
            OnProductUpdated($"Применена скидка через интерфейс");
        }
        else
        {
            Console.WriteLine($"Скидка {percentage}% недоступна через интерфейс (макс. 50%)");
        }
    }

    bool IDiscountable.IsDiscountAvailable()
    {
        return IsActive && StockQuantity > 10;
    }

    decimal IShippable.CalculateShippingCost()
    {
        decimal baseCost = 100m;
        decimal weightCost = Weight * 50m;
        return baseCost + weightCost;
    }

    string IShippable.GetShippingInfo()
    {
        var shippable = (IShippable)this;
        return $"Стоимость доставки: {shippable.CalculateShippingCost():C}\n" +
               $"Вес посылки: {Weight} кг";
    }

    void IStockable.UpdateStock(int quantity)
    {
        int oldStock = StockQuantity;
        StockQuantity += quantity;
        Console.WriteLine($"Запас {Name} обновлен: {oldStock} -> {StockQuantity} единиц");
        if (StockQuantity < 0)
        {
            StockQuantity = 0;
            Console.WriteLine("Внимание: отрицательный запас! Установлен в 0.");
        }
        OnProductUpdated($"Обновлен запас: {oldStock} -> {StockQuantity}");
    }

    bool IStockable.IsInStock()
    {
        return StockQuantity > 0 && IsActive;
    }

    int IStockable.GetStockLevel()
    {
        return StockQuantity;
    }

    private string GenerateSKU()
    {
        return $"{Manufacturer.Substring(0, Math.Min(3, Manufacturer.Length))}-" +
               $"{Name.Substring(0, Math.Min(5, Name.Length))}-" +
               $"{DateTime.Now:MMddHHmm}";
    }
}

public class Electronics : Product
{
    public int WarrantyPeriod { get; set; }
    public string PowerRequirements { get; set; }
    public bool HasBattery { get; set; }
    public string OperatingSystem { get; set; }
    public int StorageCapacity { get; set; }
    public string Connectivity { get; set; }
    public List<string> CompatibleDevices { get; set; }
    public Dictionary<string, string> TechnicalSpecs { get; set; }
    public bool IsEnergyEfficient { get; set; }
    public string EnergyClass { get; set; }

    public Electronics(string name, decimal price, string manufacturer, int warrantyPeriod) 
        : base(name, price, manufacturer)
    {
        WarrantyPeriod = warrantyPeriod;
        PowerRequirements = "100-240V";
        HasBattery = false;
        OperatingSystem = "N/A";
        StorageCapacity = 0;
        Connectivity = "Wired";
        Category = "Electronics";
        Weight = 1.0m;
        CompatibleDevices = new List<string>();
        TechnicalSpecs = new Dictionary<string, string>();
        IsEnergyEfficient = false;
        EnergyClass = "A";
        AddTag("электроника");
        AddTag("техника");
    }

    public override void Discount(decimal percentage)
    {
        if (WarrantyPeriod > 24)
        {
            decimal maxDiscount = Math.Min(percentage, 15);
            Console.WriteLine($"Товар с extended гарантией. Максимальная скидка: {maxDiscount}%");
            base.Discount(maxDiscount);
        }
        else
        {
            base.Discount(percentage);
        }
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nГарантия: {WarrantyPeriod} месяцев\n" +
               $"Питание: {PowerRequirements}\nБатарея: {(HasBattery ? "Да" : "Нет")}\n" +
               $"ОС: {OperatingSystem}\nПамять: {StorageCapacity}GB\nПодключение: {Connectivity}\n" +
               $"Энергоэффективность: {EnergyClass}\nСовместимые устройства: {CompatibleDevices.Count}";
    }

    public override void Display()
    {
        Console.WriteLine("=== ИНФОРМАЦИЯ ОБ ЭЛЕКТРОНИКЕ ===");
        Console.WriteLine(GetInfo());
        Console.WriteLine($"В наличии: {CheckAvailability()}");
        Console.WriteLine("===============================");
    }

    public void ExtendWarranty(int additionalMonths)
    {
        WarrantyPeriod += additionalMonths;
        Console.WriteLine($"Гарантия продлена на {additionalMonths} месяцев. Общий срок: {WarrantyPeriod} месяцев");
        OnProductUpdated($"Гарантия продлена на {additionalMonths} месяцев");
    }

    public decimal CalculateEnergyConsumption(int hoursPerDay)
    {
        return hoursPerDay * 0.1m * 30;
    }

    public string GetTechnicalSpecs()
    {
        return $"Технические характеристики {Name}:\n" +
               $"• Гарантия: {WarrantyPeriod} мес.\n" +
               $"• Память: {StorageCapacity}GB\n" +
               $"• ОС: {OperatingSystem}\n" +
               $"• Подключение: {Connectivity}\n" +
               $"• Энергопотребление: {EnergyClass}";
    }

    public bool SupportsWireless()
    {
        return Connectivity.ToLower().Contains("wireless") || 
               Connectivity.ToLower().Contains("bluetooth") ||
               Connectivity.ToLower().Contains("wi-fi");
    }

    public void AddCompatibleDevice(string device)
    {
        if (!CompatibleDevices.Contains(device))
        {
            CompatibleDevices.Add(device);
            OnProductUpdated($"Добавлено совместимое устройство: {device}");
        }
    }

    public bool IsCompatibleWith(string device)
    {
        return CompatibleDevices.Any(d => d.Contains(device, StringComparison.OrdinalIgnoreCase));
    }

    public void AddTechnicalSpec(string key, string value)
    {
        TechnicalSpecs[key] = value;
        OnProductUpdated($"Добавлена техническая спецификация: {key}");
    }

    public void SetEnergyEfficiency(bool isEfficient, string energyClass = "A")
    {
        IsEnergyEfficient = isEfficient;
        EnergyClass = energyClass;
        OnProductUpdated($"Обновлен класс энергоэффективности: {energyClass}");
    }
}

public class Clothing : Product
{
    public string Size { get; set; }
    public string Color { get; set; }
    public string Material { get; set; }
    public string CareInstructions { get; set; }
    public string Season { get; set; }
    public string Gender { get; set; }
    public List<string> AvailableColors { get; set; }
    public Dictionary<string, string> SizeChart { get; set; }
    public bool IsEcoFriendly { get; set; }
    public string CountryOfOrigin { get; set; }

    public Clothing(string name, decimal price, string manufacturer, string size) 
        : base(name, price, manufacturer)
    {
        Size = size;
        Color = "Black";
        Material = "Cotton";
        CareInstructions = "Machine wash";
        Season = "All season";
        Gender = "Unisex";
        Category = "Clothing";
        Weight = 0.5m;
        AvailableColors = new List<string> { "Black", "White" };
        SizeChart = new Dictionary<string, string>();
        IsEcoFriendly = false;
        CountryOfOrigin = "China";
        AddTag("одежда");
        AddTag("мода");
    }

    public override void Display()
    {
        Console.WriteLine("=== ИНФОРМАЦИЯ ОДЕЖДЫ ===");
        Console.WriteLine(GetInfo());
        Console.WriteLine($"Размер: {Size}");
        Console.WriteLine($"Цвет: {Color}");
        Console.WriteLine($"Материал: {Material}");
        Console.WriteLine($"Уход: {CareInstructions}");
        Console.WriteLine($"В наличии: {CheckAvailability()}");
        Console.WriteLine($"Доступные цвета: {AvailableColors.Count}");
        Console.WriteLine("========================");
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nРазмер: {Size}\nЦвет: {Color}\n" +
               $"Материал: {Material}\nСезон: {Season}\nПол: {Gender}\n" +
               $"Эко-френдли: {(IsEcoFriendly ? "Да" : "Нет")}\nСтрана производства: {CountryOfOrigin}";
    }

    public void UpdateCareInstructions(string newInstructions)
    {
        CareInstructions = newInstructions;
        Console.WriteLine($"Инструкции по уходу обновлены: {newInstructions}");
        OnProductUpdated("Обновлены инструкции по уходу");
    }

    public string GetSizeGuide()
    {
        return $"Руководство по размерам для {Name}:\n" +
               $"• Размер: {Size}\n" +
               $"• Материал: {Material}\n" +
               $"• Рекомендации по уходу: {CareInstructions}";
    }

    public bool IsSizeAvailable(string checkSize)
    {
        return Size.Equals(checkSize, StringComparison.OrdinalIgnoreCase) && CheckAvailability();
    }

    public decimal CalculateCustomizationCost(bool embroidery, bool printing)
    {
        decimal cost = 0;
        if (embroidery) cost += 500;
        if (printing) cost += 300;
        return cost;
    }

    public void AddAvailableColor(string color)
    {
        if (!AvailableColors.Contains(color))
        {
            AvailableColors.Add(color);
            OnProductUpdated($"Добавлен доступный цвет: {color}");
        }
    }

    public void RemoveAvailableColor(string color)
    {
        if (AvailableColors.Remove(color))
        {
            OnProductUpdated($"Удален доступный цвет: {color}");
        }
    }

    public void AddSizeToChart(string size, string measurements)
    {
        SizeChart[size] = measurements;
        OnProductUpdated($"Добавлен размер в таблицу: {size}");
    }

    public void SetEcoFriendly(bool isEcoFriendly, string countryOfOrigin = "")
    {
        IsEcoFriendly = isEcoFriendly;
        if (!string.IsNullOrEmpty(countryOfOrigin))
        {
            CountryOfOrigin = countryOfOrigin;
        }
        OnProductUpdated($"Обновлены экологические характеристики");
    }
}

public class Books : Product
{
    public string Author { get; set; }
    public string ISBN { get; set; }
    public int PageCount { get; set; }
    public string Genre { get; set; }
    public string Publisher { get; set; }
    public DateTime PublicationDate { get; set; }
    public List<string> RelatedBooks { get; set; }
    public Dictionary<string, int> ChapterLengths { get; set; }
    public bool IsBestseller { get; set; }
    public int Edition { get; set; }

    public Books(string name, decimal price, string manufacturer, string author) 
        : base(name, price, manufacturer)
    {
        Author = author;
        ISBN = GenerateISBN();
        PageCount = 0;
        Genre = "Fiction";
        Publisher = "Unknown";
        PublicationDate = DateTime.Now;
        Category = "Books";
        Weight = 0.3m;
        RelatedBooks = new List<string>();
        ChapterLengths = new Dictionary<string, int>();
        IsBestseller = false;
        Edition = 1;
        AddTag("книга");
        AddTag("литература");
    }

    public override string GetInfo()
    {
        return base.GetInfo() + $"\nАвтор: {Author}\nISBN: {ISBN}\n" +
               $"Страниц: {PageCount}\nЖанр: {Genre}\n" +
               $"Издатель: {Publisher}\nДата публикации: {PublicationDate:dd.MM.yyyy}\n" +
               $"Издание: {Edition}\nБестселлер: {(IsBestseller ? "Да" : "Нет")}";
    }

    public override void Discount(decimal percentage)
    {
        decimal maxDiscount = Math.Min(percentage, 30);
        Console.WriteLine($"Для книг максимальная скидка 30%. Установлено: {maxDiscount}%");
        base.Discount(maxDiscount);
    }

    public override void Display()
    {
        Console.WriteLine("=== ИНФОРМАЦИЯ О КНИГЕ ===");
        Console.WriteLine(GetInfo());
        Console.WriteLine($"В наличии: {CheckAvailability()}");
        Console.WriteLine($"Связанные книги: {RelatedBooks.Count}");
        Console.WriteLine("========================");
    }

    public void UpdatePublicationInfo(string publisher, DateTime publicationDate)
    {
        Publisher = publisher;
        PublicationDate = publicationDate;
        Console.WriteLine($"Информация о публикации обновлена: {publisher}, {publicationDate:dd.MM.yyyy}");
        OnProductUpdated("Обновлена информация о публикации");
    }

    public string GetReadingTimeEstimate()
    {
        if (PageCount <= 100) return "Быстрое чтение (1-2 часа)";
        if (PageCount <= 300) return "Среднее время (3-5 часов)";
        if (PageCount <= 600) return "Длительное чтение (6-10 часов)";
        return "Очень длинное чтение (10+ часов)";
    }

    public bool CheckIsBestseller()
    {
        IsBestseller = PageCount > 200 && Genre.ToLower() != "technical" && ViewCount > 100;
        return IsBestseller;
    }

    public decimal CalculateEbookPrice()
    {
        return Price * 0.6m;
    }

    public void AddRelatedBook(string bookTitle)
    {
        if (!RelatedBooks.Contains(bookTitle))
        {
            RelatedBooks.Add(bookTitle);
            OnProductUpdated($"Добавлена связанная книга: {bookTitle}");
        }
    }

    public void AddChapter(string chapterName, int pageCount)
    {
        ChapterLengths[chapterName] = pageCount;
        OnProductUpdated($"Добавлена глава: {chapterName}");
    }

    public void UpdateEdition(int newEdition)
    {
        Edition = newEdition;
        OnProductUpdated($"Обновлено издание: {newEdition}");
    }

    public int GetTotalReadingTime(int pagesPerHour = 50)
    {
        return (int)Math.Ceiling((double)PageCount / pagesPerHour);
    }

    private string GenerateISBN()
    {
        Random rand = new Random();
        return $"978-{rand.Next(1000, 9999)}-{rand.Next(100, 999)}-{rand.Next(10, 99)}";
    }
}

public class ProductCatalog
{
    private Dictionary<string, Product> _productsBySku;
    private List<Product> _allProducts;
    private HashSet<string> _categories;

    public ProductCatalog()
    {
        _productsBySku = new Dictionary<string, Product>();
        _allProducts = new List<Product>();
        _categories = new HashSet<string>();
    }

    public void AddProduct(Product product)
    {
        if (!_productsBySku.ContainsKey(product.SKU))
        {
            _productsBySku[product.SKU] = product;
            _allProducts.Add(product);
            _categories.Add(product.Category);
            product.ProductUpdated += OnProductUpdated;
            product.DiscountApplied += OnDiscountApplied;
            Console.WriteLine($"Продукт {product.Name} добавлен в каталог");
        }
    }

    public bool RemoveProduct(string sku)
    {
        if (_productsBySku.TryGetValue(sku, out var product))
        {
            _productsBySku.Remove(sku);
            _allProducts.Remove(product);
            product.ProductUpdated -= OnProductUpdated;
            product.DiscountApplied -= OnDiscountApplied;
            Console.WriteLine($"Продукт {product.Name} удален из каталог");
            return true;
        }
        return false;
    }

    public Product FindProductBySku(string sku)
    {
        return _productsBySku.GetValueOrDefault(sku);
    }

    public List<Product> FindProductsByName(string name)
    {
        return _allProducts.Where(p => p.Name.Contains(name, StringComparison.OrdinalIgnoreCase)).ToList();
    }

    public List<Product> GetProductsByCategory(string category)
    {
        return _allProducts.Where(p => p.Category.Equals(category, StringComparison.OrdinalIgnoreCase)).ToList();
    }

    public List<Product> FilterProducts(ProductFilter filter)
    {
        return _allProducts.Where(p => filter(p)).ToList();
    }

    public List<Product> GetProductsInStock()
    {
        return _allProducts.Where(p => p.CheckAvailability()).ToList();
    }

    public Dictionary<string, int> GetCategoryStats()
    {
        return _allProducts
            .GroupBy(p => p.Category)
            .ToDictionary(g => g.Key, g => g.Count());
    }

    public void ApplyDiscountToFiltered(ProductFilter filter, decimal discount)
    {
        var productsToDiscount = FilterProducts(filter);
        foreach (var product in productsToDiscount)
        {
            product.Discount(discount);
        }
    }

    public void DisplayAllProducts()
    {
        Console.WriteLine($"\n=== КАТАЛОГ ПРОДУКТОВ ({_allProducts.Count} товаров) ===");
        foreach (var product in _allProducts)
        {
            product.Display();
            Console.WriteLine();
        }
    }

    private void OnProductUpdated(Product product, string message)
    {
        Console.WriteLine($"[ОБНОВЛЕНИЕ] {product.Name}: {message}");
    }

    private void OnDiscountApplied(Product product, decimal oldPrice, decimal newPrice)
    {
        Console.WriteLine($"[СКИДКА] {product.Name}: {oldPrice:C} -> {newPrice:C} " +
                         $"(экономия: {oldPrice - newPrice:C})");
    }
}

public class ProductService
{
    private readonly ProductCatalog _catalog;

    public ProductService(ProductCatalog catalog)
    {
        _catalog = catalog;
        Product.ProductCreated += OnProductCreated;
    }

    public void AddProduct(Product product)
    {
        if (product.ValidateProduct())
        {
            _catalog.AddProduct(product);
        }
        else
        {
            Console.WriteLine($"Ошибка: Продукт {product.Name} не прошел валидацию");
        }
    }

    public void DisplayCatalogStats()
    {
        var stats = _catalog.GetCategoryStats();
        Console.WriteLine("\n=== СТАТИСТИКА КАТАЛОГА ===");
        foreach (var stat in stats)
        {
            Console.WriteLine($"{stat.Key}: {stat.Value} товаров");
        }
        var inStockCount = _catalog.GetProductsInStock().Count;
        Console.WriteLine($"Всего в наличии: {inStockCount} товаров");
    }

    public void FindAndDisplayProducts(string searchTerm)
    {
        var foundProducts = _catalog.FindProductsByName(searchTerm);
        Console.WriteLine($"\n=== РЕЗУЛЬТАТЫ ПОИСКА: '{searchTerm}' ({foundProducts.Count} найдено) ===");
        foreach (var product in foundProducts)
        {
            Console.WriteLine($"- {product.Name} ({product.Category}) - {product.Price:C}");
        }
    }

    public void DemonstrateDelegates()
    {
        Console.WriteLine("\n=== ДЕМОНСТРАЦИЯ ДЕЛЕГАТОВ ===");
        ProductFilter expensiveFilter = p => p.Price > 10000;
        ProductFilter electronicsFilter = p => p is Electronics;
        ProductFilter inStockFilter = p => p.CheckAvailability();
        var expensiveProducts = _catalog.FilterProducts(expensiveFilter);
        var electronics = _catalog.FilterProducts(electronicsFilter);
        var inStock = _catalog.FilterProducts(inStockFilter);
        Console.WriteLine($"Дорогие товары (>10,000₽): {expensiveProducts.Count}");
        Console.WriteLine($"Электроника: {electronics.Count}");
        Console.WriteLine($"Товары в наличии: {inStock.Count}");
    }

    private void OnProductCreated(Product product, string message)
    {
        Console.WriteLine($"[СОЗДАНИЕ] {message}");
    }
}

public class ProgramDemo
{
    public static void RunDemo()
    {
        Console.WriteLine("🛍️ СИСТЕМА УПРАВЛЕНИЯ ПРОДУКТАМИ С КОЛЛЕКЦИЯМИ, ДЕЛЕГАТАМИ И СОБЫТИЯМИ\n");

        ProductCatalog catalog = new ProductCatalog();
        ProductService productService = new ProductService(catalog);

        Electronics laptop = new Electronics("MacBook Pro", 150000, "Apple", 12)
        {
            Description = "Мощный ноутбук для профессионалов",
            OperatingSystem = "macOS",
            StorageCapacity = 512,
            Connectivity = "Wireless, Bluetooth, USB-C",
            HasBattery = true,
            StockQuantity = 15,
            Weight = 1.4m
        };

        Electronics phone = new Electronics("iPhone 15", 89990, "Apple", 24)
        {
            Description = "Флагманский смартфон",
            OperatingSystem = "iOS",
            StorageCapacity = 128,
            Connectivity = "Wireless, Bluetooth, 5G",
            HasBattery = true,
            StockQuantity = 8,
            Weight = 0.17m
        };

        Clothing tshirt = new Clothing("Футболка Classic", 2500, "Nike", "L")
        {
            Description = "Удобная повседневная футболка",
            Color = "White",
            Material = "100% Cotton",
            CareInstructions = "Machine wash cold",
            Season = "Summer",
            Gender = "Male",
            StockQuantity = 50,
            Weight = 0.2m
        };

        Clothing jacket = new Clothing("Куртка Winter", 12000, "The North Face", "M")
        {
            Description = "Теплая зимняя куртка",
            Color = "Black",
            Material = "Nylon",
            CareInstructions = "Dry clean only",
            Season = "Winter",
            Gender = "Unisex",
            StockQuantity = 12,
            Weight = 1.2m
        };

        Books novel = new Books("Преступление и наказание", 800, "Эксмо", "Фёдор Достоевский")
        {
            Description = "Классический роман русской литературы",
            PageCount = 672,
            Genre = "Classic Literature",
            Publisher = "Русское издательство",
            PublicationDate = new DateTime(1866, 1, 1),
            StockQuantity = 25,
            Weight = 0.6m
        };

        Books programmingBook = new Books("C# для начинающих", 1500, "O'Reilly", "Джон Смит")
        {
            Description = "Подробное руководство по C#",
            PageCount = 450,
            Genre = "Programming",
            Publisher = "O'Reilly",
            PublicationDate = new DateTime(2023, 1, 1),
            StockQuantity = 10,
            Weight = 0.8m
        };

        productService.AddProduct(laptop);
        productService.AddProduct(phone);
        productService.AddProduct(tshirt);
        productService.AddProduct(jacket);
        productService.AddProduct(novel);
        productService.AddProduct(programmingBook);

        Console.WriteLine("\n=== РАБОТА С КОЛЛЕКЦИЯМИ ===");
        laptop.AddTag("ноутбук");
        laptop.AddTag("apple");
        laptop.AddCompatibleDevice("iPhone");
        laptop.AddCompatibleDevice("iPad");
        laptop.AddTechnicalSpec("Процессор", "Apple M2");
        laptop.AddTechnicalSpec("Экран", "14.2 дюйма");
        tshirt.AddAvailableColor("Red");
        tshirt.AddAvailableColor("Blue");
        tshirt.AddSizeToChart("M", "Грудь: 100см");
        tshirt.AddSizeToChart("L", "Грудь: 105см");
        novel.AddRelatedBook("Идиот");
        novel.AddRelatedBook("Братья Карамазовы");
        novel.AddChapter("Часть 1", 150);
        novel.AddChapter("Часть 2", 200);

        Console.WriteLine("\n=== ДЕМОНСТРАЦИЯ СОБЫТИЙ ===");
        laptop.Discount(10);
        tshirt.UpdateDescription("Супер удобная футболка премиум качества");
        novel.UpdateStock(5);

        productService.DemonstrateDelegates();
        productService.FindAndDisplayProducts("Apple");
        productService.FindAndDisplayProducts("книга");
        productService.DisplayCatalogStats();
        catalog.DisplayAllProducts();

        Console.WriteLine("\n=== ДОПОЛНИТЕЛЬНЫЕ МЕТОДЫ ===");
        Console.WriteLine($"Ноутбук совместим с iPhone: {laptop.IsCompatibleWith("iPhone")}");
        Console.WriteLine($"Футболка доступна в красном: {tshirt.AvailableColors.Contains("Red")}");
        Console.WriteLine($"Время чтения книги: {novel.GetReadingTimeEstimate()}");
        Console.WriteLine($"Общее время чтения: {novel.GetTotalReadingTime()} часов");

        Console.WriteLine("\n=== ПРИМЕНЕНИЕ СКИДКИ К ФИЛЬТРУ ===");
        catalog.ApplyDiscountToFiltered(p => p is Electronics && p.Price > 50000, 15);
    }
}

ProgramDemo.RunDemo();

🛍️ СИСТЕМА УПРАВЛЕНИЯ ПРОДУКТАМИ С КОЛЛЕКЦИЯМИ, ДЕЛЕГАТАМИ И СОБЫТИЯМИ

[СОЗДАНИЕ] Создан новый продукт: MacBook Pro
[СОЗДАНИЕ] Создан новый продукт: iPhone 15
[СОЗДАНИЕ] Создан новый продукт: Футболка Classic
[СОЗДАНИЕ] Создан новый продукт: Куртка Winter
[СОЗДАНИЕ] Создан новый продукт: Преступление и наказание
[СОЗДАНИЕ] Создан новый продукт: C# для начинающих
Продукт MacBook Pro добавлен в каталог
Продукт iPhone 15 добавлен в каталог
Продукт Футболка Classic добавлен в каталог
Продукт Куртка Winter добавлен в каталог
Продукт Преступление и наказание добавлен в каталог
Продукт C# для начинающих добавлен в каталог

=== РАБОТА С КОЛЛЕКЦИЯМИ ===
[ОБНОВЛЕНИЕ] MacBook Pro: Добавлен тег: ноутбук
[ОБНОВЛЕНИЕ] MacBook Pro: Добавлен тег: apple
[ОБНОВЛЕНИЕ] MacBook Pro: Добавлено совместимое устройство: iPhone
[ОБНОВЛЕНИЕ] MacBook Pro: Добавлено совместимое устройство: iPad
[ОБНОВЛЕНИЕ] MacBook Pro: Добавлена техническая спецификация: Процессор
[ОБНОВЛЕНИЕ] MacBook Pro: Добавлена техническая 